# Swappable backends — runtime kernel selection in Python

*The Hexagonal-lite payoff, demoed from Python. `tensor.set_backend()` picks between **reference** / **eigen** / **webgpu** at runtime — the same computation routes through different `KernelBackend` adapters depending on what's installed and which one is active.*

Companion to [`tutorials/08_swappable-backends.ipynb`](../../tutorials/08_swappable-backends.ipynb) (the C++ side). Phase 6.5 per [ADR-0019](../../docs/arc42/09-decisions/0019-phase-6-5-runtime-backend-selection-via-extras.md). The Python adapter wraps the same C++ `KernelBackend` port that the C++ tutorial walks through.

## What this notebook covers

1. **The three-backend story** — what's installed vs what's active.
2. **`pip install` ergonomics** — `tensor-named-axis[eigen]` / `[webgpu]` / `[all]`.
3. **`set_backend()`, `current_backend()`, `list_available_backends()`** — the runtime API.
4. **Errors are documentation** — missing-backend `RuntimeError` includes the exact `pip install` command.
5. **Numerical agreement (QO-4)** — different backends, same answer within tolerance.
6. **Known limitation (R-P6.5.5)** — one backend per Python process.

## Prerequisites

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — `Axis`, `DynamicShape`, `DynamicTensor`, NumPy interop.

In [1]:
# Colab / Binder setup — install the tensor SDK on first run.
try:
    import tensor  # noqa: F401
except ImportError:
    import subprocess as _sp
    _sp.run(
        ["pip", "install", "--quiet",
         "git+https://github.com/uyuutosa/tensor.git"],
        check=True,
    )
    import tensor  # noqa: F401

In [2]:
import numpy as np

import tensor

print(f"tensor.__version__ = {tensor.__version__}")

tensor.__version__ = 0.1.0+dev


## 1. The three-backend story

The C++ Domain (`tensor::core`) is portable but slow on its own. Three `KernelBackend` adapters bring speed without changing the Domain:

| Backend       | What it brings                                            | When to pick it |
| ------------- | --------------------------------------------------------- | --------------- |
| `reference`   | Pure-C++; canonical answer; the test-suite oracle.       | Educational on-ramp; always works; ~5 MB wheel. |
| `eigen`       | Eigen 3.4 SIMD element-wise + BLAS-flavoured GEMM.       | CPU speedup, no GPU needed. ~20 MB wheel. |
| `webgpu`      | Dawn-backed WGSL kernels on the local GPU.                | GPU speedup. ~60 MB wheel (Dawn runtime).        |

The Python SDK loads ONE backend per process (per Phase 6.5 R-P6.5.5 — see §6 below). `list_available_backends()` tells you what's *installed*; `current_backend()` tells you what's *active*.

In [3]:
print("Installed backends:", tensor.list_available_backends())
print("Active backend:    ", tensor.current_backend())

Installed backends: ['reference']
Active backend:     reference


## 2. Install ergonomics — PEP-508 extras

The base `pip install tensor-named-axis` ships **reference only** — the smallest wheel, always works, no third-party runtime deps. To add Eigen or WebGPU, install the matching extra:

```bash
pip install tensor-named-axis              # reference only (default, ~5 MB)
pip install tensor-named-axis[eigen]       # + Eigen 3.4 (~20 MB total)
pip install tensor-named-axis[webgpu]      # + Dawn-backed WebGPU (~60 MB total)
pip install tensor-named-axis[all]         # all three
```

Each extra resolves to a companion PyPI distribution (`tensor-named-axis-eigen` / `tensor-named-axis-webgpu`) that ships a backend-specific `_tensor_native_<name>.so` into the shared `tensor/` namespace package. The maintainer's lockstep version bumper ([`tools/release.sh`](../../tools/release.sh)) keeps the three projects in sync at every tag cut.

## 3. Runtime API

Three free functions on the `tensor` module:

- `tensor.list_available_backends() -> list[str]` — sorted; subset of `["reference", "eigen", "webgpu"]`.
- `tensor.current_backend() -> str` — the loaded backend.
- `tensor.set_backend(name) -> None` — switch (subject to R-P6.5.5; see §6).

`set_backend(current_backend())` is always a safe no-op.

In [4]:
tensor.set_backend(tensor.current_backend())
print("Still active:", tensor.current_backend())

Still active: reference


## 4. Errors are documentation

If you ask for a backend that isn't installed, the error message includes the **exact `pip install` command** for the missing extra. This is the project's arc42 §8 §5.2 errors-as-docs discipline in action.

In [5]:
missing = next(
    (b for b in ("reference", "eigen", "webgpu")
     if b not in tensor.list_available_backends()),
    None,
)
if missing is None:
    print("All three backends are installed in this environment; "
          "the missing-backend error path can't be demonstrated.")
else:
    try:
        tensor.set_backend(missing)
    except RuntimeError as exc:
        print(exc)

eigen backend is not installed.
Install with:  pip install tensor-named-axis[eigen]
Or install all backends:  pip install tensor-named-axis[all]
Currently available: ['reference']


## 5. Numerical agreement (QO-4)

Whichever backend is active, the public surface produces the same result element-wise within tolerance (`1e-12` for `double` / `1e-5` for `float`). Below: a small named-axis matmul, cross-validated against `np.einsum`. The C++ side does the same QO-1 cross-validation across backends — this is the Python projection of the same guarantee.

In [6]:
rng = np.random.default_rng(seed=42)
A_np = rng.standard_normal((4, 7))
B_np = rng.standard_normal((7, 5))

A = tensor.from_numpy(A_np, ["i", "j"])
B = tensor.from_numpy(B_np, ["j", "k"])
C = tensor.contract(A, B)

expected = np.einsum("ij,jk->ik", A_np, B_np)
actual = C.numpy()
max_err = float(np.max(np.abs(actual - expected)))

print(f"Active backend:           {tensor.current_backend()}")
print(f"Max elementwise abs err:  {max_err:.2e}")
print(f"Within `double` (1e-12):  {max_err < 1e-12}")

Active backend:           reference
Max elementwise abs err:  0.00e+00
Within `double` (1e-12):  True


## 6. Known limitation — R-P6.5.5

nanobind 2.x's C++ type registry is **process-global**. Two `_tensor_native_*.so` files can't safely coexist in one Python process (the second `import` produces partial bindings). As a consequence, the Python SDK lazy-loads exactly one backend at `import tensor` time, and `set_backend()` to a different installed backend raises with the env-var workaround:

```
RuntimeError: Cannot switch from 'reference' to 'eigen' within an existing Python process —
nanobind's type registry is process-global (Phase 6.5 R-P6.5.5).
To use 'eigen' instead, restart Python with:
  TENSOR_BACKEND=eigen python ...
Or pin the backend at install time and uninstall the unwanted extra.
```

This means **switching backends requires a fresh Python process**. The patterns:

1. **Env var**: `TENSOR_BACKEND=eigen python my_script.py` selects eigen at import.
2. **Separate venvs**: keep one venv per backend if you compare them often.
3. **Subprocess**: spawn a child Python process with the env var set.

Lifting R-P6.5.5 is a Phase 7+ exploration (symbol-mangling per-backend C++ types or a pure-Python dispatch surface on top of a single native module). See arc42 §11 §5 for the risk register entry.

## 7. Where this fits

**Phase 6.5 milestone map** (2026-05-14):

| Milestone | Surface | Status |
| --------- | ------- | ------ |
| P6.5.M1 | multi-backend `_tensor_native_<name>.so` build pipeline | ✅ |
| P6.5.M2 | `set_backend()` + `current_backend()` + `list_available_backends()` Python surface | ✅ |
| P6.5.M3 | cibuildwheel extras + companion projects | ✅ |
| P6.5.M4 | `0.3.0` release ceremony | queued behind `0.2.0` |

**Sibling notebooks** (Python):

- [`00_python-sdk-tour.ipynb`](./00_python-sdk-tour.ipynb) — the surface this notebook switches between.
- [`01_python-autograd.ipynb`](./01_python-autograd.ipynb) — autograd routes through whichever backend is active.
- [`03_multifocal-tensors.ipynb`](./03_multifocal-tensors.ipynb) / [`04_python-bundle-adjustment-perspective.ipynb`](./04_python-bundle-adjustment-perspective.ipynb) — the same surface, MVG-flavour.

**Sibling notebook** (C++ xcpp20):

- [`tutorials/08_swappable-backends.ipynb`](../../tutorials/08_swappable-backends.ipynb) — the C++ side. Same `KernelBackend` port; different driving adapter.

**Cross-references**:

- [ADR-0019](../../docs/arc42/09-decisions/0019-phase-6-5-runtime-backend-selection-via-extras.md) — extras vs fat wheel decision.
- [`docs/impl-plans/2026-05-13_phase-6-5-set-backend.md`](../../docs/impl-plans/2026-05-13_phase-6-5-set-backend.md) — Phase 6.5 plan.
- [`docs/reports/2026-05-14_phase-6-5-set-backend-retrospective.md`](../../docs/reports/2026-05-14_phase-6-5-set-backend-retrospective.md) — Phase 6.5 retrospective.
- [`docs/user-manual/how-to/use-set-backend.md`](../../docs/user-manual/how-to/use-set-backend.md) — the how-to companion to this notebook.
- [`docs/arc42/11-risks/overview.md`](../../docs/arc42/11-risks/overview.md) §5 — R-P6.5.5.